<a href="https://colab.research.google.com/github/gimdiniz/datathon-fase-5-passos-magicos/blob/main/notebooks/01_visao_geral_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visão Geral da Base de Dados

## 1. Imports Iniciais

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

RAIZ_INICIAL = Path.cwd().resolve()
RAIZ = next(
    (candidato for candidato in (RAIZ_INICIAL, *RAIZ_INICIAL.parents)
     if (candidato / "src" / "carregamento_dados.py").is_file()),
    None,
)
if RAIZ is None:
    raise RuntimeError("Raiz do projeto n?o encontrada. Inicie o Jupyter dentro do reposit?rio.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.carregamento_dados import carregar_abas_base

## 2. Carregamento da Base

In [ ]:
abas_pede = carregar_abas_base()
df_2022 = abas_pede["PEDE2022"]
df_2023 = abas_pede["PEDE2023"]
df_2024 = abas_pede["PEDE2024"]

resumo_abas = pd.DataFrame(
    {
        "linhas": {aba: quadro.shape[0] for aba, quadro in abas_pede.items()},
        "colunas": {aba: quadro.shape[1] for aba, quadro in abas_pede.items()},
    }
)
resumo_abas

## 3. Dimensão das Bases

In [37]:
print(f'Número de linhas: {df_2022.shape[0]}')
print(f'Número de colunas: {df_2022.shape[1]}')
print(f'Número de linhas: {df_2023.shape[0]}')
print(f'Número de colunas: {df_2023.shape[1]}')
print(f'Número de linhas: {df_2024.shape[0]}')
print(f'Número de colunas: {df_2024.shape[1]}')

Número de linhas: 860
Número de colunas: 42
Número de linhas: 1014
Número de colunas: 48
Número de linhas: 1156
Número de colunas: 50


- Nota-se que, ao longo do tempo, a quantidade de registros aumentou, assim como o número de colunas.
- Se trata de um volume pequeno para Machine Learning. Isso facilita teste, deploy e explicação. Porém, precisamos ter cuidado com overfitting e avaliação superficial.

## 4. Limpeza e Padronização dos Dados

- Etapa de verificação e correção de nulos, ajuste de datatypes, padronização de colunas e demais ajustes.

- De acordo com a documentação, os índices IAN, IDA, IEG, IAA, IPS, IPP, IPV, INDE e PEDRA trazem informações muito relevantes para detalhar o desempenho dos alunos e sua evolução. Dessa forma, a padronização buscará preservar ao máximo essas características.

### Dados de 2022

In [4]:
df_2022.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 860 entries, 0 to 859
Data columns (total 42 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     860 non-null    object 
 1   Fase                   860 non-null    int64  
 2   Turma                  860 non-null    object 
 3   Nome                   860 non-null    object 
 4   Ano nasc               860 non-null    int64  
 5   Idade 22               860 non-null    int64  
 6   Gênero                 860 non-null    object 
 7   Ano ingresso           860 non-null    int64  
 8   Instituição de ensino  860 non-null    object 
 9   Pedra 20               323 non-null    object 
 10  Pedra 21               462 non-null    object 
 11  Pedra 22               860 non-null    object 
 12  INDE 22                860 non-null    float64
 13  Cg                     860 non-null    int64  
 14  Cf                     860 non-null    int64  
 15  Ct    

- Percebe-se que das 42 colunas, algumas possuem valores nulos: Pedra 20, Pedra 21, Avaliador3, Avaliador4, Rec Av4, Matem, Portug e Inglês.
No caso dessa fonte, entendemos que remover os nulos pode não ser a melhor alternativa, pois podemos perder informações importantes das demais colunas.

In [5]:
# Listando todas as colunas que não possuem valores nulos para referenciar posteriormente
colunas_limpas_2022 = df_2022.columns[df_2022.isna().sum() == 0].tolist()

print(f"Total de colunas sem nulos no df_2022: {len(colunas_limpas_2022)}")
print(colunas_limpas_2022)

Total de colunas sem nulos no df_2022: 34
['RA', 'Fase', 'Turma', 'Nome', 'Ano nasc', 'Idade 22', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'Pedra 22', 'INDE 22', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1', 'Rec Av1', 'Avaliador2', 'Rec Av2', 'Rec Av3', 'IAA', 'IEG', 'IPS', 'Rec Psicologia', 'IDA', 'Indicado', 'Atingiu PV', 'IPV', 'IAN', 'Fase ideal', 'Defas', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV']


- É possível notar que, das colunas de índices de desempenho, somente a coluna IPP não existe.

### Dados de 2023

In [6]:
df_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 48 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     1014 non-null   object 
 1   Fase                   1014 non-null   object 
 2   INDE 2023              931 non-null    float64
 3   Pedra 2023             931 non-null    object 
 4   Turma                  1014 non-null   object 
 5   Nome Anonimizado       1014 non-null   object 
 6   Data de Nasc           1014 non-null   object 
 7   Idade                  1014 non-null   object 
 8   Gênero                 1014 non-null   object 
 9   Ano ingresso           1014 non-null   int64  
 10  Instituição de ensino  1014 non-null   object 
 11  Pedra 20               240 non-null    object 
 12  Pedra 21               335 non-null    object 
 13  Pedra 22               600 non-null    object 
 14  Pedra 23               0 non-null      float64
 15  INDE

- Percebe-se que o df de 2023 está mais defasado, com diversas colunas com muitos ou somente nulos.

In [7]:
# Listando todas as colunas que não possuem valores nulos para referenciar posteriormente
colunas_limpas_2023 = df_2023.columns[df_2023.isna().sum() == 0].tolist()

print(f"Total de colunas sem nulos no df_2023: {len(colunas_limpas_2023)}")
print(colunas_limpas_2023)

Total de colunas sem nulos no df_2023: 12
['RA', 'Fase', 'Turma', 'Nome Anonimizado', 'Data de Nasc', 'Idade', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'IAN', 'Fase Ideal', 'Defasagem']


- Como boa parte das colunas possui algum valor nulo, inclusive de indicadores, teremos que trabalhar a remoção das linhas com registros nulos. Nesse caso, não é seguro utilizar técnicas de substituição dos nulos, dado que assumir algum tipo de desempenho pode distorcer nossas análise e, posteriormente, o modelo preditivo.

In [8]:
# Deletando linhas que possuem dados nulos em colunas mandatórias ou somente dados nulos
df_2023_ajuste = (
    df_2023
    # Remove as LINHAS onde faltam esses indicadores essenciais
    .dropna(subset=['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV', 'INDE 2023', 'Pedra 2023'])
    # Remove as COLUNAS que ficaram apenas com valores nulos
    .dropna(axis=1, how='all')
    # Reseta o índice para cobrir os buracos das linhas excluídas
    .reset_index(drop=True)
)

df_2023_ajuste.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 931 entries, 0 to 930
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     931 non-null    object 
 1   Fase                   931 non-null    object 
 2   INDE 2023              931 non-null    float64
 3   Pedra 2023             931 non-null    object 
 4   Turma                  931 non-null    object 
 5   Nome Anonimizado       931 non-null    object 
 6   Data de Nasc           931 non-null    object 
 7   Idade                  931 non-null    object 
 8   Gênero                 931 non-null    object 
 9   Ano ingresso           931 non-null    int64  
 10  Instituição de ensino  931 non-null    object 
 11  Pedra 20               220 non-null    object 
 12  Pedra 21               311 non-null    object 
 13  Pedra 22               570 non-null    object 
 14  INDE 22                570 non-null    float64
 15  Nº Av 

### Dados de 2024

In [9]:
df_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1156 entries, 0 to 1155
Data columns (total 50 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   RA                     1156 non-null   object        
 1   Fase                   1156 non-null   object        
 2   INDE 2024              1092 non-null   object        
 3   Pedra 2024             1092 non-null   object        
 4   Turma                  1156 non-null   object        
 5   Nome Anonimizado       1156 non-null   object        
 6   Data de Nasc           1156 non-null   datetime64[ns]
 7   Idade                  1156 non-null   int64         
 8   Gênero                 1156 non-null   object        
 9   Ano ingresso           1156 non-null   int64         
 10  Instituição de ensino  1155 non-null   object        
 11  Pedra 20               191 non-null    object        
 12  Pedra 21               264 non-null    object        
 13  Ped

Os dados de 2024 parecem estar mais parecidos com os de 2023, com algumas colunas com diversos nulos.

In [10]:
# Listando todas as colunas que não possuem valores nulos para referenciar posteriormente
colunas_limpas_2024 = df_2024.columns[df_2024.isna().sum() == 0].tolist()

print(f"Total de colunas sem nulos no df_2024: {len(colunas_limpas_2024)}")
print(colunas_limpas_2024)

Total de colunas sem nulos no df_2024: 15
['RA', 'Fase', 'Turma', 'Nome Anonimizado', 'Data de Nasc', 'Idade', 'Gênero', 'Ano ingresso', 'Nº Av', 'IEG', 'IAN', 'Fase Ideal', 'Defasagem', 'Ativo/ Inativo', 'Ativo/ Inativo.1']


Da mesma forma, removeremos somente linhas de colunas mandatórias com valores nulos.

In [11]:
# Deletando linhas que possuem dados nulos em colunas mandatórias
df_2024_ajuste = (
    df_2024
    # Remove as LINHAS onde faltam esses indicadores essenciais
    .dropna(subset=['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV', 'INDE 2024', 'Pedra 2024'])
    # Remove as COLUNAS que ficaram apenas com valores nulos
    .dropna(axis=1, how='all')
    # Reseta o índice para cobrir os buracos das linhas excluídas
    .reset_index(drop=True)
)
df_2024_ajuste.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1054 entries, 0 to 1053
Data columns (total 39 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   RA                     1054 non-null   object        
 1   Fase                   1054 non-null   object        
 2   INDE 2024              1054 non-null   object        
 3   Pedra 2024             1054 non-null   object        
 4   Turma                  1054 non-null   object        
 5   Nome Anonimizado       1054 non-null   object        
 6   Data de Nasc           1054 non-null   datetime64[ns]
 7   Idade                  1054 non-null   int64         
 8   Gênero                 1054 non-null   object        
 9   Ano ingresso           1054 non-null   int64         
 10  Instituição de ensino  1053 non-null   object        
 11  Pedra 20               179 non-null    object        
 12  Pedra 21               247 non-null    object        
 13  Ped

### Seleção de colunas e padronização de nomes de colunas

Utilizaremos uma função que compara as colunas dos DFs, indicando as diferenças e semelhanças entre elas.

In [12]:
def comparar_colunas_dfs(df1, df2, df3, nomes=['DF1', 'DF2', 'DF3']):
    # Transforma as listas de colunas em 'conjuntos' (sets)
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    cols3 = set(df3.columns)

    # 1. Colunas IGUAIS: Interseção (presentes nos 3 simultaneamente)
    comuns = cols1 & cols2 & cols3

    # 2. Todas as colunas existentes somadas (União)
    todas = cols1 | cols2 | cols3

    # 3. Colunas DIFERENTES: Tudo que existe, menos o que é comum a todos
    diferentes = todas - comuns

    # --- Relatório Visual ---
    print(f"Colunas presentes em TODOS ({len(comuns)}):")
    print(sorted(comuns))
    print("\n")

    print(f"Colunas DIFERENTES (não estão nos três) ({len(diferentes)}):")
    # Para facilitar, mostra em qual DF a coluna diferente está
    for col in sorted(diferentes):
        onde_tem = []
        if col in cols1: onde_tem.append(nomes[0])
        if col in cols2: onde_tem.append(nomes[1])
        if col in cols3: onde_tem.append(nomes[2])
        print(f"   - '{col}' (Presente em: {', '.join(onde_tem)})")

    return sorted(comuns), sorted(diferentes)

In [13]:
# Chamando a função e guardando os resultados em duas variáveis
colunas_iguais, colunas_diferentes = comparar_colunas_dfs(
    df1=df_2022,
    df2=df_2023_ajuste,
    df3=df_2024_ajuste,
    nomes=['2022', '2023', '2024']
)

Colunas presentes em TODOS (21):
['Ano ingresso', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Fase', 'Gênero', 'IAA', 'IAN', 'IDA', 'IEG', 'INDE 22', 'IPS', 'IPV', 'Instituição de ensino', 'Nº Av', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'RA', 'Turma']


Colunas DIFERENTES (não estão nos três) (41):
   - 'Ano nasc' (Presente em: 2022)
   - 'Atingiu PV' (Presente em: 2022)
   - 'Ativo/ Inativo' (Presente em: 2024)
   - 'Ativo/ Inativo.1' (Presente em: 2024)
   - 'Avaliador5' (Presente em: 2024)
   - 'Avaliador6' (Presente em: 2024)
   - 'Cf' (Presente em: 2022)
   - 'Cg' (Presente em: 2022)
   - 'Ct' (Presente em: 2022)
   - 'Data de Nasc' (Presente em: 2023, 2024)
   - 'Defas' (Presente em: 2022)
   - 'Defasagem' (Presente em: 2023, 2024)
   - 'Destaque IDA' (Presente em: 2022)
   - 'Destaque IEG' (Presente em: 2022)
   - 'Destaque IPV' (Presente em: 2022)
   - 'Escola' (Presente em: 2024)
   - 'Fase Ideal' (Presente em: 2023, 2024)
   - 'Fase ideal' (Presente em: 2022)
   - '

É necessário alterar o nome de algumas colunas, que existem para os três anos, mas estão com nomes divergentes (por exemplo, Defas e Defasagem).
No entanto, notamos que existem as colunas [Data de Nasc] em 2023 e 2024, e [Ano nasc] 2022, que potencialmente tem padrões diferentes de dados.

In [ ]:
df_comparacao = pd.concat([
    df_2022['Ano nasc'].rename('Nascimento (Base 2022)'),
    df_2023_ajuste['Data de Nasc'].rename('Nascimento (Base 2023)'),
    df_2024_ajuste['Data de Nasc'].rename('Nascimento (Base 2024)')
], axis=1)

resumo_nascimento_original = pd.DataFrame({
    "tipo": df_comparacao.dtypes.astype(str),
    "percentual_ausencia": df_comparacao.isna().mean().mul(100).round(2),
    "valores_distintos": df_comparacao.nunique(dropna=True),
})
resumo_nascimento_original

In [15]:
#Extraindo o ano para normalização dos dados de nascimento
#em 2023, os dados estão no padrão mês/dia
df_2023_ajuste['Data de Nasc'] = pd.to_datetime(df_2023_ajuste['Data de Nasc'], format='%m/%d/%Y', errors='coerce').dt.year

#já em 2024, seguem o padrão dia/mês
df_2024_ajuste['Data de Nasc'] = pd.to_datetime(df_2024_ajuste['Data de Nasc'], format='%d/%m/%Y', errors='coerce').dt.year

In [ ]:
df_comparacao = pd.concat([
    df_2022['Ano nasc'].rename('Nascimento (Base 2022)'),
    df_2023_ajuste['Data de Nasc'].rename('Nascimento (Base 2023)'),
    df_2024_ajuste['Data de Nasc'].rename('Nascimento (Base 2024)')
], axis=1)

resumo_nascimento_normalizado = pd.DataFrame({
    "tipo": df_comparacao.dtypes.astype(str),
    "percentual_ausencia": df_comparacao.isna().mean().mul(100).round(2),
    "valores_distintos": df_comparacao.nunique(dropna=True),
})
resumo_nascimento_normalizado

In [17]:
# Definindo as regras para a base de 2022
dic_2022 = {
    'Fase ideal': 'Fase Ideal',
    'Matem': 'Mat',
    'Portug': 'Por',
    'Inglês': 'Ing',
    'Defas': 'Defasagem',
    'Nome': 'Nome Anonimizado',
    'Idade 22': 'Idade',
    'Pedra 22' : 'Pedra' ,
    'Ano nasc': 'Ano Nascimento' ,
    'INDE 22' : 'INDE'
}

# Definindo as regras para a base de 2023
dic_2023 = {
    'INDE 2023': 'INDE',
    # 'INDE 23': 'INDE',
    'Pedra 2023': 'Pedra',
    # 'Pedra 23': 'Pedra',
    'Data de Nasc': 'Ano Nascimento'
}

# Definindo as regras para a base de 2024
dic_2024 = {
    'INDE 2024': 'INDE',
    # 'INDE 23': 'INDE',
    'Pedra 2024': 'Pedra',
    'Data de Nasc': 'Ano Nascimento'
}

# Aplicando a transformação
df_2022 = df_2022.rename(columns=dic_2022)
df_2023_ajuste = df_2023_ajuste.rename(columns=dic_2023)
df_2024_ajuste = df_2024_ajuste.rename(columns=dic_2024)

In [18]:
#Rodando novamente o comparativo entre colunas nos dfs

colunas_iguais, colunas_diferentes = comparar_colunas_dfs(
    df1=df_2022,
    df2=df_2023_ajuste,
    df3=df_2024_ajuste,
    nomes=['2022', '2023', '2024']
)

Colunas presentes em TODOS (29):
['Ano Nascimento', 'Ano ingresso', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Defasagem', 'Fase', 'Fase Ideal', 'Gênero', 'IAA', 'IAN', 'IDA', 'IEG', 'INDE', 'IPS', 'IPV', 'Idade', 'Ing', 'Instituição de ensino', 'Mat', 'Nome Anonimizado', 'Nº Av', 'Pedra', 'Pedra 20', 'Pedra 21', 'Por', 'RA', 'Turma']


Colunas DIFERENTES (não estão nos três) (23):
   - 'Atingiu PV' (Presente em: 2022)
   - 'Ativo/ Inativo' (Presente em: 2024)
   - 'Ativo/ Inativo.1' (Presente em: 2024)
   - 'Avaliador5' (Presente em: 2024)
   - 'Avaliador6' (Presente em: 2024)
   - 'Cf' (Presente em: 2022)
   - 'Cg' (Presente em: 2022)
   - 'Ct' (Presente em: 2022)
   - 'Destaque IDA' (Presente em: 2022)
   - 'Destaque IEG' (Presente em: 2022)
   - 'Destaque IPV' (Presente em: 2022)
   - 'Escola' (Presente em: 2024)
   - 'INDE 22' (Presente em: 2023, 2024)
   - 'INDE 23' (Presente em: 2024)
   - 'IPP' (Presente em: 2023, 2024)
   - 'Indicado' (Presente em: 2022)
   - 'Pe

Para garantir uma análise sólida, vamos focar em avaliar indicadores em comum entre os três anos, além do indicador IPP, que é necessário para análise posterior de defasagem dos alunos.

In [19]:
# 1. Descobre a interseção (as colunas que existem nos 3 anos)
colunas_comuns = sorted(set(df_2022.columns) & set(df_2023_ajuste.columns) & set(df_2024_ajuste.columns))

# 2. Cria uma segunda lista para 23/24 incluindo IPP
colunas_com_ipp = colunas_comuns + ['IPP']

print(f"Total de colunas em comum (2022): {len(colunas_comuns)}")
print(f"Total de colunas com IPP (2023 e 2024): {len(colunas_com_ipp)}")

# 3. Filtra base a base
df_22_comum = df_2022[colunas_comuns].copy()
df_23_comum = df_2023_ajuste[colunas_com_ipp].copy()
df_24_comum = df_2024_ajuste[colunas_com_ipp].copy()

Total de colunas em comum (2022): 29
Total de colunas com IPP (2023 e 2024): 30


In [20]:
colunas_iguais, colunas_diferentes = comparar_colunas_dfs(
    df1=df_22_comum,
    df2=df_23_comum,
    df3=df_24_comum,
    nomes=['2022', '2023', '2024']
)

Colunas presentes em TODOS (29):
['Ano Nascimento', 'Ano ingresso', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Defasagem', 'Fase', 'Fase Ideal', 'Gênero', 'IAA', 'IAN', 'IDA', 'IEG', 'INDE', 'IPS', 'IPV', 'Idade', 'Ing', 'Instituição de ensino', 'Mat', 'Nome Anonimizado', 'Nº Av', 'Pedra', 'Pedra 20', 'Pedra 21', 'Por', 'RA', 'Turma']


Colunas DIFERENTES (não estão nos três) (1):
   - 'IPP' (Presente em: 2023, 2024)


### Corrigindo Datatypes

In [21]:
def comparar_dtypes_dfs(df1, df2, df3, nomes=['2022', '2023', '2024']):
    # Cria um DataFrame onde cada coluna é o tipo de dado de um ano específico
    df_comparativo = pd.DataFrame({
        nomes[0]: df1.dtypes,
        nomes[1]: df2.dtypes,
        nomes[2]: df3.dtypes
    })

    # Cria uma regra para checar se o tipo do Ano 1 é igual ao Ano 2 E igual ao Ano 3
    # Se forem todos iguais, 'Divergente' será False. Se houver diferença, será True.
    df_comparativo['Divergente'] = ~(
        (df_comparativo[nomes[0]] == df_comparativo[nomes[1]]) &
        (df_comparativo[nomes[1]] == df_comparativo[nomes[2]])
    )

    # --- Relatório Visual ---
    divergencias = df_comparativo[df_comparativo['Divergente'] == True]

    print("RELATÓRIO DE COMPATIBILIDADE DE TIPOS")
    print("-" * 50)

    if len(divergencias) == 0:
        print("Todas as colunas compartilham os mesmos tipos de dados nos 3 anos.")
    else:
        print(f"Encontradas {len(divergencias)} colunas com tipos divergentes")
        print(divergencias.drop(columns=['Divergente']))

    return df_comparativo

In [22]:
relatorio_tipos = comparar_dtypes_dfs(df_22_comum, df_23_comum, df_24_comum)

RELATÓRIO DE COMPATIBILIDADE DE TIPOS
--------------------------------------------------
Encontradas 6 colunas com tipos divergentes
                   2022     2023     2024
Ano Nascimento    int64    int32    int32
Fase              int64   object   object
INDE            float64  float64   object
IPP                 NaN  float64  float64
Idade             int64   object    int64
Nº Av             int64  float64    int64


In [ ]:
# CORREÇÃO 2022
df_22_comum['Fase'] = df_22_comum['Fase'].astype(str)  # Garante texto para o mapeamento nominal
df_22_comum['Idade'] = df_22_comum['Idade'].astype('Int64')
df_22_comum['Ano Nascimento'] = df_22_comum['Ano Nascimento'].astype('Int64')
df_22_comum['Nº Av'] = df_22_comum['Nº Av'].astype('Int64')

# CORREÇÃO 2023
df_23_comum['Fase'] = df_23_comum['Fase'].astype(str)
df_23_comum['Idade'] = pd.to_numeric(df_23_comum['Idade'], errors='coerce').astype('Int64')
df_23_comum['Ano Nascimento'] = df_23_comum['Ano Nascimento'].astype('Int64')
df_23_comum['Nº Av'] = df_23_comum['Nº Av'].astype('Int64')

# CORREÇÃO 2024
df_24_comum['INDE'] = pd.to_numeric(df_24_comum['INDE'], errors='coerce')
df_24_comum['Fase'] = df_24_comum['Fase'].astype(str)
df_24_comum['Idade'] = df_24_comum['Idade'].astype('Int64')
df_24_comum['Ano Nascimento'] = df_24_comum['Ano Nascimento'].astype('Int64')
df_24_comum['Nº Av'] = df_24_comum['Nº Av'].astype('Int64')

# Identifica explicitamente o ciclo antes da concatena??o longitudinal
df_22_comum['Ano_Referencia'] = 2022
df_23_comum['Ano_Referencia'] = 2023
df_24_comum['Ano_Referencia'] = 2024

relatorio_tipos = comparar_dtypes_dfs(df_22_comum, df_23_comum, df_24_comum)

### Unificando as bases em um único df

In [ ]:
df_concat = pd.concat([df_22_comum, df_23_comum, df_24_comum], ignore_index=True)
resumo_estrutura_consolidada = pd.DataFrame({
    "metrica": ["linhas", "colunas"],
    "valor": [df_concat.shape[0], df_concat.shape[1]],
})
resumo_estrutura_consolidada

### Verificando nulos e normalizando nomes e ordens de colunas

In [25]:
df_concat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2845 entries, 0 to 2844
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano Nascimento         2845 non-null   Int64  
 1   Ano ingresso           2845 non-null   int64  
 2   Avaliador1             2820 non-null   object 
 3   Avaliador2             2820 non-null   object 
 4   Avaliador3             2027 non-null   object 
 5   Avaliador4             1045 non-null   object 
 6   Defasagem              2845 non-null   int64  
 7   Fase                   2845 non-null   object 
 8   Fase Ideal             2845 non-null   object 
 9   Gênero                 2845 non-null   object 
 10  IAA                    2845 non-null   float64
 11  IAN                    2845 non-null   float64
 12  IDA                    2845 non-null   float64
 13  IEG                    2845 non-null   float64
 14  INDE                   2845 non-null   float64
 15  IPS 

Para os dados de Idade faltantes, vamos calcular considerando o Ano_Referencia - Ano Nascimento, pois é possível inferir.
As notas ausentes serão preservadas: interpolar a base concatenada poderia transferir informação entre alunos ou anos.
O registro faltante de Instituição de Ensino será marcado como "Não informado", sem inventar uma escola.


In [26]:
# Preencher a Idade faltante (Ano Referência - Ano Nascimento)
df_concat['Idade'] = df_concat['Idade'].fillna(df_concat['Ano_Referencia'] - df_concat['Ano Nascimento'])

# Renomear as colunas de Notas
df_concat = df_concat.rename(columns={
    'Mat': 'Nota_Matematica',
    'Por': 'Nota_Portugues'
})

# Preservar notas ausentes evita interpolação entre alunos ou períodos distintos

# Preencher somente a ausência categórica, sem atribuir uma escola não observada
df_concat['Instituição de ensino'] = df_concat['Instituição de ensino'].fillna('Não informado')

# Removendo a coluna Nº Av, pois não é possível inferir seu significado sem registro dela no dicionário de dados
if 'Nº Av' in df_concat.columns:
    df_concat = df_concat.drop(columns=['Nº Av'])

# 1. Identifica todas as colunas que ainda têm valores nulos
colunas_com_nulos = df_concat.columns[df_concat.isna().any()].tolist()

# As colunas são preservadas; ausência pode ser estrutural e será tratada por análise
print('Colunas que permanecem com valores nulos:')
print(colunas_com_nulos)

# Verificação final
df_concat.info()

Colunas que permanecem com valores nulos:
['Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Ing', 'Nota_Matematica', 'Pedra 20', 'Pedra 21', 'Nota_Portugues', 'IPP']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2845 entries, 0 to 2844
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano Nascimento         2845 non-null   Int64  
 1   Ano ingresso           2845 non-null   int64  
 2   Avaliador1             2820 non-null   object 
 3   Avaliador2             2820 non-null   object 
 4   Avaliador3             2027 non-null   object 
 5   Avaliador4             1045 non-null   object 
 6   Defasagem              2845 non-null   int64  
 7   Fase                   2845 non-null   object 
 8   Fase Ideal             2845 non-null   object 
 9   Gênero                 2845 non-null   object 
 10  IAA                    2845 non-null   float64
 11  IAN                    2845 non-null  

### Verificar Duplicados

In [27]:
#Checagem geral de duplicados
df_concat.duplicated().sum()

np.int64(0)

In [28]:
#Transforma RA e Nome Anonimizado em texto e remove espaços em branco inúteis no início e no fim, para garantir padronização
df_concat['RA'] = df_concat['RA'].astype(str).str.strip()
df_concat['Nome Anonimizado'] = df_concat['Nome Anonimizado'].astype(str).str.strip()

In [ ]:
# Checagem de duplicidade por RA no mesmo ano, apresentada somente de forma agregada
duplicados_mesmo_ano = df_concat[df_concat.duplicated(subset=['RA', 'Ano_Referencia'], keep=False)]
resumo_duplicados_ano = (
    duplicados_mesmo_ano.groupby('Ano_Referencia').size()
    .reindex(sorted(df_concat['Ano_Referencia'].unique()), fill_value=0)
    .rename('registros_duplicados')
    .to_frame()
)
resumo_duplicados_ano

In [ ]:
# Checagem longitudinal agregada, sem exibir identificadores
historico_alunos_repetidos = df_concat[df_concat.duplicated(subset=['RA'], keep=False)]
resumo_historico_repetido = historico_alunos_repetidos.groupby('Ano_Referencia').agg(
    registros=('RA', 'size'),
    alunos_unicos=('RA', 'nunique'),
)
resumo_historico_repetido

In [ ]:
# Distribui??o agregada da quantidade de anos observados por aluno
alunos_frequencia = df_concat['RA'].value_counts()
distribuicao_frequencia = (
    alunos_frequencia.value_counts()
    .sort_index()
    .rename_axis('anos_observados')
    .rename('quantidade_alunos')
    .to_frame()
)
distribuicao_frequencia

- O conjunto de dados não contêm dados 100% duplicados, nem registros duplicados de um mesmo aluno em um ano. Porém, nota-se que existe duplicação entre anos, ou seja, um mesmo aluno foi registrado nas bases de 2022, 2023 e 2024. Isso não indica a necessidade de normalização, mas é um grande ponto de atenção para a etapa de treinamento de modelo.

### Valores únicos

Como última checagem antes de finalizar a análise, pro caso de termos algum valor não normalizados no dataset.

In [ ]:
# Resumo de qualidade sem imprimir valores ou registros individuais
resumo_qualidade = pd.DataFrame({
    'tipo': df_concat.dtypes.astype(str),
    'percentual_ausencia': df_concat.isna().mean().mul(100).round(2),
    'valores_distintos': df_concat.nunique(dropna=True),
})
resumo_qualidade

Ainda se faz necessária a normalização da coluna Gênero, Fase e Pedras:

In [33]:
# 1. NORMALIZAÇÃO DO GÊNERO

mapeamento_genero = {
    'Menina': 'Feminino',
    'Menino': 'Masculino'
}

# Aplica a substituição
df_concat['Gênero'] = df_concat['Gênero'].replace(mapeamento_genero)

print("--- Distribuição de Gênero ---")
print(df_concat['Gênero'].value_counts())
print("\n")


# 2. NORMALIZAÇÃO DA FASE

def padronizar_fase(valor):
    # Transforma em string, joga pra maiúsculo e tira espaços em branco nas pontas
    texto = str(valor).upper().strip()

    # Tratamento específico para o nível Alfa
    if texto in ['0', 'ALFA']:
        return 'ALFA'

    # Busca o primeiro dígito numérico na string (ex: "1A" -> "1", "FASE 3" -> "3")
    match = re.search(r'\d', texto)
    if match:
        numero = match.group(0)
        return f'FASE {numero}'

    # Retorno de segurança caso venha algum lixo na base que não seja Alfa nem número
    return texto

# 3. NORMALIZAÇÃO DAS PEDRAS

mapeamento_pedras = {
    'Agata': 'Ágata'
}

# Aplica a substituição
df_concat['Pedra'] = df_concat['Pedra'].replace(mapeamento_pedras)

print("--- Distribuição de Pedras ---")
print(df_concat['Pedra'].value_counts())
print("\n")


# Aplica a função linha a linha na coluna Fase
df_concat['Fase'] = df_concat['Fase'].apply(padronizar_fase)

print("--- Distribuição de Fases ---")
print(df_concat['Fase'].value_counts().sort_index())

--- Distribuição de Gênero ---
Gênero
Feminino     1519
Masculino    1326
Name: count, dtype: int64


--- Distribuição de Pedras ---
Pedra
Ametista    1120
Ágata        721
Topázio      688
Quartzo      316
Name: count, dtype: int64


--- Distribuição de Fases ---
Fase
ALFA      617
FASE 1    550
FASE 2    539
FASE 3    491
FASE 4    279
FASE 5    225
FASE 6     76
FASE 7     68
Name: count, dtype: int64


## 5. Exportando a Base Tratada

In [38]:
# Define o path de saída
processed_path = RAIZ / "data/processed"

# Cria a pasta/path se ele não existir
processed_path.mkdir(parents=True, exist_ok=True)

# Exporta a base tratada
df_concat.to_csv(processed_path / "passos_magicos_clean_eda.csv", index=False, encoding="utf-8")

In [39]:
# Verificando se a base foi salva corretamente
os.listdir(processed_path)

['base_2024_clean.csv', 'passos_magicos_clean_eda.csv']